# SIMT 编程实战：Transpose 算子基础实现

## 概述

前面《编程模型》章节我们系统学习了 SIMT 的硬件架构、线程架构、核函数、内存层级、同步机制和编程 API。本节进入**实战**：用 Ascend C SIMT 从零实现一个二维 **Transpose（矩阵转置）算子**，走完 Device 端核函数、Host 端调用、编译运行的完整流程。

Transpose 用于交换二维矩阵的行和列，是矩阵计算、特征布局转换、图像处理和算子融合中常见的数据重排操作。它的计算逻辑很简单：每个输入元素只需要移动到输出矩阵中的一个确定位置，因此很适合作为 SIMT 入门实战算子。

本节只介绍最朴素、最容易理解的实现：**每个 SIMT 线程处理一个元素，直接从 Global Memory 读取输入并写回转置后的输出地址**。这种写法便于理解 `blockIdx`、`threadIdx`、`blockDim` 和二维地址展开规则。第 7 章会继续围绕同一个 Transpose 算子展开性能优化，逐步解决本节基础实现中暴露出的 Thread Block 超发、GM 非连续写、UB 中转、bank 冲突和双缓冲等问题。

### 学习前置要求

学习本节前，建议已经具备以下基础：

- 已学习《编程模型》中 SIMT 的核函数、线程索引、内存层级等内容。
- 了解 C/C++ 中一维数组表示二维矩阵的基本方式。
- 已具备在 NPU 环境中使用 CMake + 毕昇编译器编译 Ascend C 样例的基础。

### 学习目标

完成本节后，开发者应能够：

- 说明 Transpose 算子的功能和二维到一维的地址映射关系。
- 用 SIMT 思路设计 Transpose 的数据切分：一个线程处理一个矩阵元素。
- 写出最基础的 Transpose Device 端核函数，理解线程全局下标、边界检查和输出地址计算。
- 写出 Host 端代码，完成 ACL 初始化、Device 内存管理、核函数启动、结果拷回和正确性校验。
- 用 CMake + 毕昇编译器编译并运行一个 SIMT 算子。
- 理解本节基础实现与第 7 章 Transpose 性能优化实践之间的衔接关系。

### 本节内容

- 环境准备
- Transpose 算子功能介绍
- 算子设计：数据切分
- Device 端核函数实现
- Host 端调用实现
- 测试数据与验证
- 编译运行与基础性能观察
- 课后编程习题


## 1. 环境准备

正式开始学习之前，先要对 Jupyter 环境进行初始化。以下代码完成初始化，并将 CANN 环境变量导入当前 Jupyter 进程，保证后续能够使用毕昇编译器完成算子的开发与编译。

本节所有代码生成、编译和运行都在 `Sources/04.06` 目录下进行，`src` 目录仅作为只读的源码仓库，任何单元格都不会写入 `src`。


In [ ]:
import os
import subprocess
from pathlib import Path

candidate_env_scripts = [
    "/home/wulinyu/cann_pkg/0701/cann-9.1.0/set_env.sh",
    os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh",
    "/usr/local/Ascend/cann/set_env.sh",
]
set_env = next((p for p in candidate_env_scripts if Path(p).exists()), candidate_env_scripts[-1])

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("
"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

WORKSPACE = Path("Sources/04.06")
WORKSPACE.mkdir(parents=True, exist_ok=True)

print("Environment initialization process completed successfully.")
print(f"Workspace: {WORKSPACE.resolve()}")


## 2. Transpose 算子功能介绍

Transpose 用于交换二维矩阵的行和列。设输入矩阵形状为 `height x width`，输出矩阵形状为 `width x height`，计算公式如下：

```text
output(col, row) = input(row, col)
```

实际计算中，`input` 和 `output` 都按一维数组连续存储。对于输入矩阵中的 `input(row, col)`，其一维下标为：

```cpp
input_index = row * width + col;
```

转置后，该元素写入输出矩阵中的 `output(col, row)`。因为输出矩阵每行有 `height` 个元素，所以输出一维下标为：

```cpp
output_index = col * height + row;
```

下图以 `4 x 3` 输入矩阵为例，展示 Transpose 后行列维度和元素排列的变化：

![](../07_advanced_operator_practice/images/07_05_simt_transpose/transpose_example.png)

本节实现固定使用 `1024 x 1024` 的 `float` 矩阵，输入和输出元素总数都是 `1024 * 1024`。后续第 7 章会继续使用 Transpose 算子作为优化对象，在本节基础实现之上逐步引入更高性能的实现方式。

| 项目 | 取值 |
| --- | --- |
| 输入形状 | `1024 x 1024` |
| 输出形状 | `1024 x 1024` |
| 输入数据类型 | `float` |
| 输出数据类型 | `float` |
| 核函数名 | `transpose_basic_kernel` |
| 基础实现策略 | 每个线程处理一个元素，直接 GM 读写 |


## 3. 算子设计：数据切分

Transpose 的每个输出元素只依赖一个输入元素，各元素之间没有数据依赖。因此，最简单的 SIMT 切分方式是：**一个线程处理输入矩阵中的一个元素**。

设当前线程的全局线性下标为 `idx`：

```cpp
uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
```

再把 `idx` 还原成输入矩阵中的二维坐标：

```cpp
uint32_t row = idx / width;
uint32_t col = idx - row * width;
```

最后根据转置规则写入输出位置：

```cpp
output[col * height + row] = input[idx];
```

本例输入元素总数为 `1024 * 1024 = 1048576`。由于每个线程只做一次读取、一次地址计算和一次写回，单线程工作量很轻，因此每个 Thread Block 启动 `2048` 个线程，对应 Thread Block 数为：

```cpp
num_blocks = (total_length + THREAD_COUNT - 1) / THREAD_COUNT;
```

对于 `1024 x 1024` 输入，`num_blocks = 512`。这个配置足够直接地展示 SIMT 编程模型，但它不是高性能最终形态：第 7 章会继续分析为什么这种写法会带来 Thread Block 超发和 GM 非连续写问题，并给出逐步优化路径。


In [ ]:
!mkdir -p Sources/04.06/transpose_basic

接下来逐段把代码写入工作目录下的 `Sources/04.06/transpose_basic/transpose_basic.asc`。第一个 `%%writefile` 覆盖创建文件，后续 `%%writefile -a` 追加。

先写入头文件、常量和函数声明：


In [ ]:
%%writefile Sources/04.06/transpose_basic/transpose_basic.asc
#include <algorithm>
#include <cmath>
#include <iostream>
#include <iterator>
#include <vector>
#include "acl/acl.h"
#include "simt_api/asc_simt.h"

constexpr uint32_t THREAD_COUNT = 2048;


## 4. Device 端核函数实现

Device 端核函数 `transpose_basic_kernel` 是本节最核心的部分。它遵循 SIMT 核函数的常见结构：

1. 通过 `blockIdx.x * blockDim.x + threadIdx.x` 计算线程的全局线性下标。
2. 通过 `if (idx >= total_length)` 做边界检查，避免最后一个 Thread Block 线程数不足时越界访问。
3. 将线性下标还原成输入矩阵的 `(row, col)` 坐标。
4. 按 `output(col, row) = input(row, col)` 写入输出矩阵。

把核函数追加写入 `transpose_basic.asc`：


In [ ]:
%%writefile -a Sources/04.06/transpose_basic/transpose_basic.asc

__global__ __launch_bounds__(THREAD_COUNT) void transpose_basic_kernel(
    float* output, const float* input, uint32_t width, uint32_t height, uint32_t total_length)
{
    uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= total_length) {
        return;
    }

    uint32_t row = idx / width;
    uint32_t col = idx - row * width;
    output[col * height + row] = input[idx];
}


## 5. Host 端调用实现

Host 端负责完成 ACL 初始化、内存申请、数据拷贝、核函数启动和结果回收。本节把这部分封装成 `transpose_basic` 函数，主要步骤如下：

1. 根据 `height * width` 计算元素总数和字节数。
2. 初始化 ACL，选择 Device，创建 Stream。
3. 分配 Device 端输入、输出内存，以及 Host 端输出缓存。
4. 将输入数据从 Host 拷贝到 Device。
5. 使用 `transpose_basic_kernel<<<grid, block, 0, stream>>>(...)` 启动核函数。
6. 同步 Stream，等待 Device 端执行完成。
7. 将输出从 Device 拷回 Host，并释放相关资源。

下面把 Host 调用函数追加写入 `transpose_basic.asc`：


In [ ]:
%%writefile -a Sources/04.06/transpose_basic/transpose_basic.asc

std::vector<float> transpose_basic(const std::vector<float>& input, uint32_t height, uint32_t width)
{
    uint32_t total_length = height * width;
    size_t byte_size = static_cast<size_t>(total_length) * sizeof(float);

    aclInit(nullptr);
    int32_t device_id = 0;
    aclrtSetDevice(device_id);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    float* input_device = nullptr;
    float* output_device = nullptr;
    float* output_host = nullptr;

    aclrtMalloc(reinterpret_cast<void**>(&input_device), byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(reinterpret_cast<void**>(&output_device), byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMallocHost(reinterpret_cast<void**>(&output_host), byte_size);

    aclrtMemcpy(input_device, byte_size, input.data(), byte_size, ACL_MEMCPY_HOST_TO_DEVICE);

    uint32_t blocks_per_grid = (total_length + THREAD_COUNT - 1) / THREAD_COUNT;
    dim3 grid(blocks_per_grid, 1, 1);
    dim3 block(THREAD_COUNT, 1, 1);
    transpose_basic_kernel<<<grid, block, 0, stream>>>(output_device, input_device, width, height, total_length);

    aclrtSynchronizeStream(stream);
    aclrtMemcpy(output_host, byte_size, output_device, byte_size, ACL_MEMCPY_DEVICE_TO_HOST);

    std::vector<float> output(output_host, output_host + total_length);

    aclrtFree(input_device);
    aclrtFree(output_device);
    aclrtFreeHost(output_host);
    aclrtDestroyStream(stream);
    aclrtResetDevice(device_id);
    aclFinalize();

    return output;
}


## 6. 测试数据与验证

为了验证算子正确性，Host 侧还需要两个辅助函数：

- `transpose_golden`：在 CPU 上按同样的转置公式计算预期结果。
- `verify_result`：逐元素比较 NPU 输出和 CPU 预期结果。

`main` 函数构造 `1024 x 1024` 测试矩阵，调用 `transpose_basic` 得到 NPU 结果，并与 `golden` 对比。

把验证代码和 `main` 函数追加写入 `transpose_basic.asc`，至此整个源文件组装完成：


In [ ]:
%%writefile -a Sources/04.06/transpose_basic/transpose_basic.asc

void transpose_golden(const std::vector<float>& input, std::vector<float>& golden, uint32_t height, uint32_t width)
{
    for (uint32_t row = 0; row < height; ++row) {
        for (uint32_t col = 0; col < width; ++col) {
            golden[col * height + row] = input[row * width + col];
        }
    }
}

uint32_t verify_result(const std::vector<float>& output, const std::vector<float>& golden)
{
    auto print_tensor = [](const std::vector<float>& tensor, const char* name) {
        constexpr size_t max_print_size = 20;
        std::cout << name << ": ";
        std::copy(
            tensor.begin(), tensor.begin() + std::min(tensor.size(), max_print_size),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > max_print_size) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    print_tensor(output, "Output");
    print_tensor(golden, "Golden");

    for (size_t i = 0; i < output.size(); ++i) {
        if (std::fabs(output[i] - golden[i]) > 1e-3f) {
            std::cout << "[Failed] Case accuracy verification failed!" << std::endl;
            std::cout << "First mismatch index: " << i
                      << ", output: " << output[i]
                      << ", golden: " << golden[i] << std::endl;
            return 1;
        }
    }

    std::cout << "[Success] Case accuracy verification passed." << std::endl;
    return 0;
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t matrix_height = 1024;
    constexpr uint32_t matrix_width = 1024;
    constexpr uint32_t total_length = matrix_height * matrix_width;

    std::vector<float> input(total_length);
    for (uint32_t i = 0; i < total_length; ++i) {
        input[i] = static_cast<float>(i) * 1.25f;
    }

    std::vector<float> golden(total_length);
    transpose_golden(input, golden, matrix_height, matrix_width);

    std::vector<float> output = transpose_basic(input, matrix_height, matrix_width);
    return verify_result(output, golden);
}


## 7. 编译与运行

用 CMake + 毕昇编译工具链编译。关键是声明 `ASC` 语言、设置 NPU 架构，并加上 `--enable-simt` 启用 SIMT 编程场景。运行下方单元格生成 `CMakeLists.txt`：


In [ ]:
%%writefile Sources/04.06/transpose_basic/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

find_package(ASC REQUIRED)
project(simt_transpose_basic LANGUAGES ASC CXX)

add_executable(demo
    transpose_basic.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)


**编译选项说明：**

| 选项 | 说明 |
| --- | --- |
| `--npu-arch=dav-3510` | 指定 NPU 架构版本，`dav-` 后为架构号，Ascend 950PR/Ascend 950DT 对应 `dav-3510` |
| `--enable-simt` | 启用 SIMT 编程场景，编译 SIMT 算子必须添加 |

在 `Sources/04.06/transpose_basic` 目录下执行编译运行三连：


In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/04.06/transpose_basic && mkdir -p build && cd build &&  cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j &&  ./demo


编译运行成功后，`main` 会构造测试数据、调用算子、并和 CPU 算出的 `golden` 对比，预期输出包含：

```text
[Success] Case accuracy verification passed.
```


### 使用 msopprof 采集基础实现性能

完成正确性验证后，可以使用 `msopprof` 采集当前基础实现的性能数据：


In [ ]:
# 采集最基础 GM 直接读写版本的算子性能
!cd Sources/04.06/transpose_basic/build && msopprof ./demo


本节的基础实现追求的是**先跑通开发流程**，不是最终性能。它的访存模式是：

```text
GM 连续读 + GM 非连续写
```

同一个 Warp 内相邻线程读取 `input[idx]` 时地址连续，但写入 `output[col * height + row]` 时会跨行跳转，GM 写地址不连续。同时，`1024 x 1024` 输入在 `THREAD_COUNT = 2048` 时会启动 512 个 Thread Block，可能远超设备物理核数，产生调度排队开销。

这些问题会在第 7 章《SIMT Transpose 算子优化实践》中继续展开：先限制 Thread Block 数，再引入 UB 中转、padding 和双缓冲，逐步把本节的最简单实现优化成更高性能版本。


## 小结

本节用 Transpose 算子完整走了一遍 SIMT 算子开发流程，把前面《编程模型》学到的概念落到了实处：

- **算子语义**：`output(col, row) = input(row, col)`，核心是二维坐标和一维地址之间的映射。
- **线程划分**：一个线程处理一个输入元素，用全局线程下标 `idx` 定位数据。
- **核函数结构**：计算全局下标、做边界检查、还原二维坐标、写入转置后的输出地址。
- **Host 调用**：完成 ACL 初始化、Device 内存分配、Host/Device 数据拷贝、核函数启动、同步和结果校验。
- **编译运行**：用 CMake + 毕昇编译器，关键编译选项是 `--npu-arch` 指定架构、`--enable-simt` 启用 SIMT。

掌握这个最小可运行样例后，第 7 章会继续沿用同一个 Transpose 算子分析性能瓶颈，让 04 章的“能跑通”自然衔接到 07 章的“跑得快”。


## 课后编程习题：矩形矩阵 Transpose 算子

本节已经实现了 `1024 x 1024` 方阵的 Transpose。请参考《SIMT 线程架构》《SIMT 核函数》《SIMT 内存层级》的内容，自行实现一个**矩形矩阵 Transpose 算子**。

本练习的输入矩阵形状为 `513 x 1025`，输出矩阵形状为 `1025 x 513`。由于元素总数不能被每个 Thread Block 的线程数整除，Device 端核函数必须正确处理边界检查。

| 项目 | 取值 |
| --- | --- |
| 输入形状 | `513 x 1025` |
| 输出形状 | `1025 x 513` |
| 输入数据类型 | `float` |
| 输出数据类型 | `float` |
| 核函数名 | `transpose_rect_custom` |
| 要求 | 每个线程处理一个元素，并通过 `if (idx >= total_length)` 防止越界 |

请补全下面代码中 Device 端 `transpose_rect_custom` 函数体里的 TODO。


In [ ]:
!mkdir -p Sources/04.06/transpose_rect

In [ ]:
%%writefile Sources/04.06/transpose_rect/transpose_rect.asc
#include <algorithm>
#include <cmath>
#include <iostream>
#include <iterator>
#include <vector>
#include "acl/acl.h"
#include "simt_api/asc_simt.h"

constexpr uint32_t THREAD_COUNT = 1024;

__global__ __launch_bounds__(THREAD_COUNT) void transpose_rect_custom(
    float* output, const float* input, uint32_t width, uint32_t height, uint32_t total_length)
{
    uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= total_length) {
        return;
    }

    // TODO: 1. 根据 idx、width 计算输入矩阵中的 row 和 col
    // TODO: 2. 根据 output(col, row) = input(row, col) 写入 output
}

void transpose_golden(const std::vector<float>& input, std::vector<float>& golden, uint32_t height, uint32_t width)
{
    for (uint32_t row = 0; row < height; ++row) {
        for (uint32_t col = 0; col < width; ++col) {
            golden[col * height + row] = input[row * width + col];
        }
    }
}

uint32_t verify_result(const std::vector<float>& output, const std::vector<float>& golden)
{
    auto print_tensor = [](const std::vector<float>& tensor, const char* name) {
        constexpr size_t max_print_size = 20;
        std::cout << name << ": ";
        std::copy(
            tensor.begin(), tensor.begin() + std::min(tensor.size(), max_print_size),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > max_print_size) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    print_tensor(output, "Output");
    print_tensor(golden, "Golden");

    for (size_t i = 0; i < output.size(); ++i) {
        if (std::fabs(output[i] - golden[i]) > 1e-3f) {
            std::cout << "[Failed] Case accuracy verification failed!" << std::endl;
            std::cout << "First mismatch index: " << i
                      << ", output: " << output[i]
                      << ", golden: " << golden[i] << std::endl;
            return 1;
        }
    }

    std::cout << "[Success] Case accuracy verification passed." << std::endl;
    return 0;
}

std::vector<float> transpose_rect(const std::vector<float>& input, uint32_t height, uint32_t width)
{
    uint32_t total_length = height * width;
    size_t byte_size = static_cast<size_t>(total_length) * sizeof(float);

    aclInit(nullptr);
    int32_t device_id = 0;
    aclrtSetDevice(device_id);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    float* input_device = nullptr;
    float* output_device = nullptr;
    float* output_host = nullptr;

    aclrtMalloc(reinterpret_cast<void**>(&input_device), byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(reinterpret_cast<void**>(&output_device), byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMallocHost(reinterpret_cast<void**>(&output_host), byte_size);

    aclrtMemcpy(input_device, byte_size, input.data(), byte_size, ACL_MEMCPY_HOST_TO_DEVICE);

    uint32_t blocks_per_grid = (total_length + THREAD_COUNT - 1) / THREAD_COUNT;
    dim3 grid(blocks_per_grid, 1, 1);
    dim3 block(THREAD_COUNT, 1, 1);
    transpose_rect_custom<<<grid, block, 0, stream>>>(output_device, input_device, width, height, total_length);

    aclrtSynchronizeStream(stream);
    aclrtMemcpy(output_host, byte_size, output_device, byte_size, ACL_MEMCPY_DEVICE_TO_HOST);

    std::vector<float> output(output_host, output_host + total_length);

    aclrtFree(input_device);
    aclrtFree(output_device);
    aclrtFreeHost(output_host);
    aclrtDestroyStream(stream);
    aclrtResetDevice(device_id);
    aclFinalize();

    return output;
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t matrix_height = 513;
    constexpr uint32_t matrix_width = 1025;
    constexpr uint32_t total_length = matrix_height * matrix_width;

    std::vector<float> input(total_length);
    for (uint32_t i = 0; i < total_length; ++i) {
        input[i] = static_cast<float>(i % 997) * 0.5f;
    }

    std::vector<float> golden(total_length);
    transpose_golden(input, golden, matrix_height, matrix_width);

    std::vector<float> output = transpose_rect(input, matrix_height, matrix_width);
    return verify_result(output, golden);
}


再创建 CMake 配置：


In [ ]:
%%writefile Sources/04.06/transpose_rect/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

find_package(ASC REQUIRED)
project(simt_transpose_rect LANGUAGES ASC CXX)

add_executable(demo
    transpose_rect.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)


完成 `transpose_rect.asc` 后，在 NPU 环境中执行以下命令验证结果：


In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/04.06/transpose_rect && mkdir -p build && cd build &&  cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j &&  ./demo


如果实现正确，输出应包含：

```text
[Success] Case accuracy verification passed.
```

完成后，可以执行下面的单元格查看参考实现。建议先独立完成，再对照参考答案检查线程索引、边界检查和转置地址计算。


In [ ]:
!cat answer/04_06_simt_transpose_operator/transpose_rect.asc

In [ ]:
!cat answer/04_06_simt_transpose_operator/CMakeLists.txt

完成本练习后，可以对比方阵 Transpose 和矩形矩阵 Transpose：二者的线程划分方式完全相同，都是一个线程处理一个元素；不同点在于输出矩阵的行宽从 `width` 变成了 `height`，因此写回地址必须始终使用 `col * height + row`。这个练习可以帮助你把“二维坐标还原”和“转置后地址映射”两个关键点彻底区分开。
